# 03 — Baseline Training

**Goal:** Train a minimal semantic segmentation CNN from scratch to verify the full pipeline runs end-to-end.

---

## What model are we using?

This notebook uses a **very simple 3-layer CNN** as a placeholder.  
Its purpose is to confirm that:
- Data loading works correctly.
- Tensor shapes are correct at every step.
- The loss goes down (even slightly) after one epoch.
- Checkpointing saves and restores weights.

This model will **not** produce good segmentations — that is expected.  
Once the pipeline is verified, replace the model with U-Net, FCN, or DeepLabv3.

> **Prerequisite:** Run notebooks 00–02 first and ensure `pairs` is populated.

In [1]:
import os
import json
import hashlib
import platform
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from src.data_paths import RAW_DATA_DIR, MODELS_DIR, LOGS_DIR, ensure_project_dirs
from src.dataset import AI4MarsDataset, find_image_files, find_mask_files, build_pairs_by_stem, split_pairs_by_source_stem
from src.train_utils import get_device, save_checkpoint, train_one_epoch, evaluate

ensure_project_dirs()

print(f"PyTorch version: {torch.__version__}")

All project directories are ready.
PyTorch version: 2.12.1+cu126


## Step 1 — Load Pairs

In [2]:
DATA_ROOT = RAW_DATA_DIR  # adjust if needed

image_files = find_image_files(DATA_ROOT)
mask_files  = find_mask_files(DATA_ROOT)
pairs       = build_pairs_by_stem(image_files, mask_files)

print(f"Total pairs (all schemes): {len(pairs)}")

# Restrict baseline training to NAV labels only.
# M2020_GEO masks contain different class IDs (e.g. 20, 30, 50) that are
# incompatible with a 4-class NAV setup (NUM_CLASSES=4).
nav_pairs = [(img, msk) for img, msk in pairs if "M2020_GEO" not in str(msk)]
pairs = nav_pairs
print(f"Total pairs (NAV only)   : {len(pairs)}")

if len(pairs) == 0:
    raise RuntimeError(
        "No NAV image/mask pairs found. "
        "Ensure AI4Mars data is extracted under data/raw/ and labels are present."
    )


Total pairs (all schemes): 46460
Total pairs (NAV only)   : 37958


## Step 2 — Hyperparameters

In [3]:
IMAGE_SIZE   = (256, 256)   # (width, height) — PIL convention
BATCH_SIZE   = 4
NUM_CLASSES  = 4            # soil, bedrock, sand, big_rock
IGNORE_INDEX = 255          # unlabeled pixels in AI4Mars masks
LEARNING_RATE = 1e-3
NUM_EPOCHS   = 3            # quick quality check for weighted + pretrained baseline
VAL_SPLIT    = 0.1          # 10% of data held out for validation

## Step 3 — Create Train/Val Split and DataLoaders

In [4]:
TRAIN_SPLIT = 0.70
VAL_SPLIT = 0.15
TEST_SPLIT = 0.15

splits = split_pairs_by_source_stem(
    pairs,
    train_ratio=TRAIN_SPLIT,
    val_ratio=VAL_SPLIT,
    test_ratio=TEST_SPLIT,
    seed=42,
)

train_pairs = splits["train"]
val_pairs = splits["val"]
test_pairs = splits["test"]

train_dataset = AI4MarsDataset(train_pairs, image_size=IMAGE_SIZE)
val_dataset = AI4MarsDataset(val_pairs, image_size=IMAGE_SIZE)
test_dataset = AI4MarsDataset(test_pairs, image_size=IMAGE_SIZE)

print(f"Train samples: {len(train_dataset)}")
print(f"Val samples  : {len(val_dataset)}")
print(f"Test samples : {len(test_dataset)}")

split_fingerprint_payload = "\n".join(
    f"{img}|{msk}" for img, msk in val_pairs
)
split_fingerprint = hashlib.sha256(split_fingerprint_payload.encode("utf-8")).hexdigest()

split_metadata = {
    "split_strategy": "grouped_source_stem",
    "seed": 42,
    "train_ratio": TRAIN_SPLIT,
    "val_ratio": VAL_SPLIT,
    "test_ratio": TEST_SPLIT,
    "nav_only": True,
    "val_pair_count": len(val_pairs),
    "val_split_fingerprint_sha256": split_fingerprint,
}

# DataLoader performance knobs:
# - Windows/CPU often plateaus with too many workers due to process overhead.
cpu_count = os.cpu_count() or 2
if torch.cuda.is_available():
    NUM_WORKERS = max(2, min(8, cpu_count - 1))
else:
    NUM_WORKERS = min(4, max(1, cpu_count // 2))

PIN_MEMORY = torch.cuda.is_available()

print(f"DataLoader num_workers: {NUM_WORKERS}")
print(f"DataLoader pin_memory : {PIN_MEMORY}")

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=NUM_WORKERS > 0,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=NUM_WORKERS > 0,
)

print("Split strategy: deterministic source-stem grouping with fixed seed (42)")
print(f"Split ratios : train={TRAIN_SPLIT:.2f}, val={VAL_SPLIT:.2f}, test={TEST_SPLIT:.2f}")
print(f"Validation split fingerprint (sha256): {split_fingerprint}")

Train samples: 26538
Val samples  : 5716
Test samples : 5704
DataLoader num_workers: 7
DataLoader pin_memory : True
Split strategy: deterministic source-stem grouping with fixed seed (42)
Split ratios : train=0.70, val=0.15, test=0.15
Validation split fingerprint (sha256): 16f0185c372ff26067f04911125cfe025e9f8620e7c5231e2046b522104fef80


## Step 3.5 — Compute Data-Driven Class Weights

Estimate class frequencies from the training split so the loss reflects the
actual NAV label distribution instead of an approximate hand-tuned vector.

In [5]:
stats_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=NUM_WORKERS > 0,
)

class_pixel_counts = torch.zeros(NUM_CLASSES, dtype=torch.long)
for _, masks in stats_loader:
    valid = masks != IGNORE_INDEX
    for class_id in range(NUM_CLASSES):
        class_pixel_counts[class_id] += ((masks == class_id) & valid).sum()

total_labeled_pixels = class_pixel_counts.sum().item()
class_frequencies = class_pixel_counts.float() / max(total_labeled_pixels, 1)
class_weights = 1.0 / torch.clamp(class_frequencies, min=1e-8)
class_weights = class_weights / class_weights.mean()

print("Train-split labeled pixel counts by class:")
for class_id, class_name in enumerate(["soil", "bedrock", "sand", "big_rock"]):
    print(
        f"  class {class_id} ({class_name:>10s}): {class_pixel_counts[class_id].item():>12d} pixels "
        f"({class_frequencies[class_id].item() * 100:6.2f}%)"
    )

print("\nEmpirical class weights (inverse-frequency, mean-normalized):")
for class_id, class_name in enumerate(["soil", "bedrock", "sand", "big_rock"]):
    print(f"  class {class_id} ({class_name:>10s}): {class_weights[class_id].item():.4f}")

Train-split labeled pixel counts by class:
  class 0 (      soil):    443964309 pixels ( 39.40%)
  class 1 (   bedrock):    322461623 pixels ( 28.61%)
  class 2 (      sand):    310687633 pixels ( 27.57%)
  class 3 (  big_rock):     49793490 pixels (  4.42%)

Empirical class weights (inverse-frequency, mean-normalized):
  class 0 (      soil): 0.3144
  class 1 (   bedrock): 0.4329
  class 2 (      sand): 0.4493
  class 3 (  big_rock): 2.8034


## Step 4 — Verify Batch Shapes

Always check tensor shapes **before** training.  
A shape mismatch will cause a confusing error inside the model.

In [6]:
images, masks = next(iter(train_loader))
print(f"Image batch shape : {images.shape}   (expected [B, 3, H, W])")
print(f"Mask  batch shape : {masks.shape}    (expected [B, H, W])")
print(f"Image dtype       : {images.dtype}   (expected float32)")
print(f"Mask  dtype       : {masks.dtype}    (expected int64 / long)")
print(f"Image value range : [{images.min():.3f}, {images.max():.3f}]  (expected [0, 1])")

Image batch shape : torch.Size([4, 3, 256, 256])   (expected [B, 3, H, W])
Mask  batch shape : torch.Size([4, 256, 256])    (expected [B, H, W])
Image dtype       : torch.float32   (expected float32)
Mask  dtype       : torch.int64    (expected int64 / long)
Image value range : [0.000, 1.000]  (expected [0, 1])


## Step 5 — Define a Stronger Pretrained Segmentation Model

For the first real baseline, use a pretrained U-Net encoder so the model starts with useful low-level visual features (edges, textures) instead of learning everything from scratch.

In [7]:
import segmentation_models_pytorch as smp

# Use a lighter encoder on CPU to keep iterations practical.
device = get_device()
encoder_name = "mobilenet_v2" if device.type == "cpu" else "resnet34"

model = smp.Unet(
    encoder_name=encoder_name,
    encoder_weights="imagenet",
    in_channels=3,
    classes=NUM_CLASSES,
).to(device)

# Quick shape check before training
dummy = torch.randn(2, 3, IMAGE_SIZE[1], IMAGE_SIZE[0]).to(device)
out = model(dummy)
print(f"Encoder: {encoder_name}")
print(f"Model output shape: {out.shape}   (expected [2, {NUM_CLASSES}, {IMAGE_SIZE[1]}, {IMAGE_SIZE[0]}])")

num_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {num_params:,}")

Using device: cuda
Encoder: resnet34
Model output shape: torch.Size([2, 4, 256, 256])   (expected [2, 4, 256, 256])
Model parameters: 24,436,804


## Loss Function and Optimiser

In [8]:
# Weighted cross-entropy to counter class imbalance using the empirical class
# distribution computed above.
weights = class_weights.to(device)
loss_fn = nn.CrossEntropyLoss(weight=weights, ignore_index=IGNORE_INDEX)

optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
print(f"Class weights: {weights.tolist()}")

Class weights: [0.31441888213157654, 0.4328910708427429, 0.44929611682891846, 2.803393840789795]


## Reproducibility Log

Record the key settings needed to reproduce the baseline experiment and the
saved checkpoint.

In [9]:
reproducibility_log = {
    "hardware": {
        "device": str(device),
        "platform": platform.platform(),
        "python": platform.python_version(),
        "torch": torch.__version__,
        "cuda_available": torch.cuda.is_available(),
        "cuda_device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
    },
    "image_size": IMAGE_SIZE,
    "batch_size": BATCH_SIZE,
    "model": {
        "type": "U-Net",
        "encoder": encoder_name,
        "encoder_weights": "imagenet",
        "num_classes": NUM_CLASSES,
    },
    "optimizer": {
        "name": "Adam",
        "learning_rate": LEARNING_RATE,
    },
    "epochs": NUM_EPOCHS,
    "seed": 42,
    "split_strategy": "deterministic source-stem grouping",
    "split_metadata": split_metadata,
    "dataset_version": "ai4mars-dataset-merged-0.6 (Zenodo 15995036)",
    "checkpoint_name": "weighted_unet_epoch03.pth",
}

reproducibility_path = LOGS_DIR / "reproducibility_log.json"
with open(reproducibility_path, "w", encoding="utf-8") as f:
    json.dump(reproducibility_log, f, indent=2)

print(f"Saved reproducibility log to {reproducibility_path}")
print(json.dumps(reproducibility_log, indent=2))

Saved reproducibility log to C:\Users\Jacob\AI4Mars\outputs\logs\reproducibility_log.json
{
  "hardware": {
    "device": "cuda",
    "platform": "Windows-11-10.0.26200-SP0",
    "python": "3.14.3",
    "torch": "2.12.1+cu126",
    "cuda_available": true,
    "cuda_device": "NVIDIA GeForce GTX 1080 Ti"
  },
  "image_size": [
    256,
    256
  ],
  "batch_size": 4,
  "model": {
    "type": "U-Net",
    "encoder": "resnet34",
    "encoder_weights": "imagenet",
    "num_classes": 4
  },
  "optimizer": {
    "name": "Adam",
    "learning_rate": 0.001
  },
  "epochs": 3,
  "seed": 42,
  "split_strategy": "deterministic source-stem grouping",
  "split_metadata": {
    "split_strategy": "grouped_source_stem",
    "seed": 42,
    "train_ratio": 0.7,
    "val_ratio": 0.15,
    "test_ratio": 0.15,
    "nav_only": true,
    "val_pair_count": 5716,
    "val_split_fingerprint_sha256": "16f0185c372ff26067f04911125cfe025e9f8620e7c5231e2046b522104fef80"
  },
  "dataset_version": "ai4mars-dataset-merg

## Training Loop

In [10]:
CLASS_NAMES = ["soil", "bedrock", "sand", "big_rock"]

for epoch in range(1, NUM_EPOCHS + 1):
    print(f"\nEpoch {epoch}/{NUM_EPOCHS}")
    print("-" * 40)

    train_loss = train_one_epoch(model, train_loader, optimizer, loss_fn, device)
    val_metrics = evaluate(
        model,
        val_loader,
        loss_fn,
        device,
        num_classes=NUM_CLASSES,
        ignore_index=IGNORE_INDEX,
        return_per_class_iou=(epoch == NUM_EPOCHS),  # avoid extra overhead each epoch
    )

    print(f"  Train loss : {train_loss:.4f}")
    print(f"  Val loss   : {val_metrics['val_loss']:.4f}")
    print(f"  Pixel acc  : {val_metrics['pixel_acc']:.4f}")
    print(f"  Mean IoU   : {val_metrics['mean_iou']:.4f}")

    per_class_iou = val_metrics.get("per_class_iou")
    if per_class_iou is not None:
        print("  Per-class IoU:")
        valid_scores = []
        for class_name, score in zip(CLASS_NAMES, per_class_iou):
            if score is None:
                print(f"    - {class_name:8s}: n/a")
            else:
                print(f"    - {class_name:8s}: {score:.4f}")
                valid_scores.append((class_name, score))

        if valid_scores:
            hardest_class, hardest_score = min(valid_scores, key=lambda x: x[1])
            print(f"  Hardest class this epoch: {hardest_class} (IoU={hardest_score:.4f})")


Epoch 1/3
----------------------------------------
  Batch 10/6635  loss=1.4729
  Batch 20/6635  loss=1.4308
  Batch 30/6635  loss=1.4506
  Batch 40/6635  loss=2.0817
  Batch 50/6635  loss=1.4747
  Batch 60/6635  loss=1.1918
  Batch 70/6635  loss=1.6715
  Batch 80/6635  loss=1.4580
  Batch 90/6635  loss=1.2195
  Batch 100/6635  loss=1.1738
  Batch 110/6635  loss=1.3524
  Batch 120/6635  loss=1.3793
  Batch 130/6635  loss=1.2471
  Batch 140/6635  loss=1.0766
  Batch 150/6635  loss=1.3206
  Batch 160/6635  loss=1.1593
  Batch 170/6635  loss=1.3510
  Batch 180/6635  loss=1.5304
  Batch 190/6635  loss=1.0439
  Batch 200/6635  loss=0.8914
  Batch 210/6635  loss=0.9727
  Batch 220/6635  loss=1.3923
  Batch 230/6635  loss=1.1206
  Batch 240/6635  loss=1.1930
  Batch 250/6635  loss=1.4983
  Batch 260/6635  loss=0.9453
  Batch 270/6635  loss=0.9284
  Batch 280/6635  loss=1.3528
  Batch 290/6635  loss=1.3293
  Batch 300/6635  loss=1.5028
  Batch 310/6635  loss=0.9872
  Batch 320/6635  loss=1.17

## Step 8 — Save Checkpoint

In [11]:
checkpoint_path = MODELS_DIR / "weighted_unet_epoch03.pth"
save_checkpoint(
    model,
    optimizer,
    epoch=NUM_EPOCHS,
    path=checkpoint_path,
    metadata=split_metadata,
 )
print(f"Saved checkpoint to {checkpoint_path}")

Checkpoint saved -> C:\Users\Jacob\AI4Mars\models\weighted_unet_epoch03.pth
Saved checkpoint to C:\Users\Jacob\AI4Mars\models\weighted_unet_epoch03.pth


## Next Steps

- Open `04_evaluation_error_analysis.ipynb` to load the checkpoint and analyse predictions.
- Run a cleaner experiment series next:
    - **Baseline:** U-Net with a ResNet34 encoder.
    - **Alternative encoder:** U-Net with EfficientNet if it fits local hardware.
    - **Alternative decoder family:** DeepLabV3+ for stronger context aggregation.
    - **Loss experiments:** Dice, Focal, or CE+Dice hybrids.